# Heavy detection signals on a Colab T4 (Phase 3)

Runs the signals that do not fit on the dev laptop: **CometKiwi**, **xCOMET-XL** (gated: accept both licenses on Hugging Face first), plus the model-based signals on a GPU.

1. Runtime > Change runtime type > **T4 GPU**.
2. Upload `mt-meaning-loss.zip` (the project folder without `.venv/` and `data/raw/`) to the Colab files pane, or mount Drive.
3. Add your token as a Colab *secret* named `HF_TOKEN` (key icon in the sidebar). It is read below and never printed.

In [ ]:
!unzip -q -o mt-meaning-loss.zip -d /content && cd /content/mt-meaning-loss && pip install -q -r requirements.txt -r requirements-colab.txt && pip install -q -e . --no-deps

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # never print this
os.chdir('/content/mt-meaning-loss')
!python -c "import torch; print(torch.cuda.get_device_name(0))"

In [ ]:
# Generate the data on Colab too (small, deterministic): FLORES-200 + synthetic errors
!mml download-data --pair en-hi --what flores && mml synth --pair en-hi
!mml download-data --pair en-de --what flores && mml synth --pair en-de

In [ ]:
# All six signals on 60 validation examples per pair; prints mean features per gold label.
# Signals that cannot run are listed as SKIPPED with the reason (nothing is faked).
!mml signals --pair en-hi --profile colab_t4 --split val --limit 60
!mml signals --pair en-de --profile colab_t4 --split val --limit 60

In [ ]:
# Download the caches so Phase 4 can reuse them locally (data/cache/signals.sqlite, data/cache/mt.sqlite)
!zip -q -r signal_caches.zip data/cache results/runs
from google.colab import files; files.download('signal_caches.zip')